# Scenario 1 — Minitron: Quick & Reliable Compression (~1h45 on 2x H200)

This notebook prunes Qwen3-8B down to ~7B parameters using **Minitron** (homogeneous pruning), then distills and evaluates the result.

**Pipeline:** Prune → Evaluate → Distill → Export → Evaluate

**Prerequisites:**
- Run [`00_prerequisites.ipynb`](00_prerequisites.ipynb) first to prepare the distillation dataset.
- Base model downloaded at `/workspace/models/Qwen3-8B`.

## 1. Prune (long step)

Minitron's `prune_minitron.py` script handles the full pruning pipeline in one command:
1. Loads the model into Megatron-Bridge format
2. Runs a calibration pass on the [Nemotron-Post-Training-Dataset-v2](https://huggingface.co/datasets/nvidia/Nemotron-Post-Training-Dataset-v2) to compute importance scores
3. Enumerates all valid (depth, width) configurations, keeps the 10 candidates with parameter count closest to and below 7B, scores them using a fast MMLU proxy, and selects the best one
4. Exports the pruned model as a HuggingFace checkpoint

We skip pruning `num_attention_heads` to keep the GQA structure intact (the model reduces hidden size, FFN intermediate size, and drops layers instead).

> **Calibration dataset note:** The pruning script automatically downloads and uses 1,024 samples from `nvidia/Nemotron-Post-Training-Dataset-v2` for calibration (configurable via `--calib_dataset_name` and `--calib_num_samples`).

In [ ]:
!cd /workspace/Model-Optimizer/examples/megatron_bridge && \
torchrun --nproc_per_node 2 prune_minitron.py \
    --pp_size 2 \
    --hf_model_name_or_path /workspace/models/Qwen3-8B \
    --prune_target_params 7e9 \
    --hparams_to_skip num_attention_heads \
    --output_hf_path /workspace/output/Qwen3-8B-Pruned-7B

**Expected output:** Minitron selects the following best architecture:

```
[BEST SUBNET] {'num_layers': 32, 'hidden_size': 3840, 'ffn_hidden_size': 12288} -> 6.96B params, 0.7073 score

Dropping decoder layers [28, 31, 32, 33] from model.
```

The model goes from 36 to 32 layers (dropping 4 late layers), hidden size is reduced from 4096 to 3840, and FFN intermediate size stays at 12288.

## 2. Verify pruned model

Check that the pruned checkpoint was saved correctly.

In [ ]:
!ls -lh /workspace/output/Qwen3-8B-Pruned-7B/

## 3. Evaluate pruned model (before distillation)

Run MMLU (5-shot) on the pruned model to measure how much accuracy was lost during pruning. This gives us the pre-distillation baseline.

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=/workspace/output/Qwen3-8B-Pruned-7B,dtype=bfloat16 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4

## 4. Distill

Run knowledge distillation: the pruned model (student) learns to mimic the original Qwen3-8B (teacher) using logits-based KL divergence loss on the tokenized [WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext/tree/main/wikitext-103-v1) dataset.

We run 100 iterations with a sequence length of 4096 and a global batch size of 4 (~1.6M tokens).

Launch TensorBoard to monitor the distillation loss in real time. Open http://localhost:6006 in your browser once the distillation cell is running.

> **Tip:** In the TensorBoard settings (top-right gear icon), check **"Reload data"** so the charts update automatically as training progresses.

In [ ]:
import subprocess

subprocess.Popen(
    ["tensorboard", "--logdir", "/workspace/output/distill_output_7B/tb_logs", "--port", "6006"]
)
print("TensorBoard started at http://localhost:6006")

Now, let's run the distillation.
> **Expected runtime: ~20-30 minutes on 2x H200.**

In [ ]:
!cd /workspace/Model-Optimizer/examples/megatron_bridge && \
torchrun --nnodes 1 --nproc_per_node=2 distill.py \
    --student_hf_path /workspace/output/Qwen3-8B-Pruned-7B \
    --teacher_hf_path /workspace/models/Qwen3-8B \
    --data_paths 1.0 /workspace/datasets/wikitext-103-v1/wikitext-train_text \
    --output_dir /workspace/output/distill_output_7B \
    --seq_length 4096 \
    --tp_size 2 \
    --pp_size 1 \
    --mbs 1 \
    --gbs 4 \
    --train_iters 100 \
    --lr 0.0001 \
    --min_lr 1e-05 \
    --lr_warmup_iters 10 \
    --eval_interval 10 \
    --eval_iters 10 \
    --log_interval 1

Finally, kill tensorboard:

In [ ]:
subprocess.run(["pkill", "-f", "tensorboard"])

**Expected distillation loss curve:**

![Minitron 7B distillation loss](distillation_loss_7B.png)

The KL divergence drops from ~0.93 to ~0.38 over 100 iterations, with training and validation loss tracking closely (no overfitting).

## 5. Export to HuggingFace format

Distillation saves a Megatron distributed checkpoint. Convert it back to HuggingFace format for evaluation and deployment.

In [ ]:
!python /opt/Megatron-Bridge/examples/conversion/convert_checkpoints.py export \
    --hf-model /workspace/output/Qwen3-8B-Pruned-7B \
    --megatron-path /workspace/output/distill_output_7B/checkpoints/iter_0000100 \
    --hf-path /workspace/output/distilled_Qwen3-8B-Pruned-7B

## 6. Evaluate distilled model

Run MMLU (5-shot) on the distilled model. Compare with the pre-distillation score from Step 3 to measure distillation recovery.

**Expected results on Qwen3-8B:**

| Model | MMLU (5-shot) | % of Teacher |
|---|---|---|
| Qwen3-8B (teacher) | 0.7493 | 100% |
| Minitron — pruned | 0.7038 | 93.9% |
| Minitron — pruned + distilled | **0.7166** | **95.6%** |

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=/workspace/output/distilled_Qwen3-8B-Pruned-7B,dtype=bfloat16 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4